In [ ]:
import sys
sys.path.insert(0, '../..')

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import edunn as nn
from edunn import utils

In [ ]:
np.set_printoptions(threshold=sys.maxsize)

# 1D Convolutional Layer

Convolution is a fundamental mathematical operation in signal and system processing. It is widely used to filter signals, smooth data and detect features.

In the context of signal processing, 1D convolution is used to extract certain features from the signal, such as edges in the case of images, or to remove noise.

$$
y[n] = (x \circledast w) = \sum_{k} w[k] \cdot x[n+k]
$$

> NOTE: below we explain how to build the ``Conv2d`` layer; its 1D version can be interpreted as a particular case of it, in which the height dimension is always 1, which implies that the _stride_ and _padding_ are 1 and 0 respectively in that case.

# 2D Convolutional Layer

Following the same idea, we can extend the concept of convolution to matrices, that is, a convolution in 2 dimensions. This is useful for grayscale images, RGB images and batches of them.

## Forward Method

A 2D convolutional layer is built by applying a 2D convolution operation to the input, followed by the addition of a Bias if desired. Convolution is an operation on two functions $x$ and $w$ that produces a third function, which can be interpreted as a version of the input $x$ "filtered" by $w$ (also known as the kernel). 

* Although convolution is defined in continuous form, what interests us here is its discrete version.

So, given a batch of input images with shape $(N,C_{\text{in}},H,W)$ and a batch of filters with shape $(M,C_{\text{in}},F,F)$, the convolution operation can be written as:

$$
y[l,m,i,j] = (x \circledast w) = \sum_{a}\sum_{b} w[m,:,a,b] \cdot x[l,:,i+a,j+b]
$$

Where $N$ is the number of images in the batch, $C_{\text{in/out}}$ is the number of channels of the input image and of the output feature map (respectively), $M$ is the number of desired feature maps and $F$ is the kernel size.

In this formula you can see that the sums run over the kernel dimensions and over the $C$ channels at the same time; it also assumes that the **stride** is 1 and the **padding** is 0. The bias term is added after the convolution operation.

* Activation functions can operate in a matrix fashion, applied to each value $y[k,i,j]$ without needing any change.

|||
|:-:|:-:|
|![conv2d_forward.gif](img/conv2d_forward.gif)|![conv2d_multilayer.gif](img/conv2d_multilayer.gif)|


### Hyperparameters

The size of the resulting matrix $y$ is strictly determined by the following parameters:

* `kernel_size`: the size of the filter being used.
* `stride`: the number of steps the filter jumps each time it is applied. 
* `padding`: the number of zero-filled pixels added at the borders.
   * Applying the filter in a discrete fashion causes two problems:
       * Loss of information at the borders.
       * Reduction of the final size of the vector.
   <!-- * It is used to obtain a resulting image of a desired size. -->

|strides|padding|
|:-:|:-:|
|![conv2d_strides.gif](img/conv2d_strides.gif)|![conv2d_padding.gif](img/conv2d_padding.gif)|

### Shapes

The size of the output $y$ (that is, of the resulting feature maps) depends on the size $H_{in} \times W_{in}$ of the input image, the kernel size $F$, the stride $S$ and the padding $P$. The general formula for computing the height $H_{out}$ and the width $W_{out}$ of the output of a convolutional layer is:

$$
\begin{aligned}
A_{out} &= \left\lfloor \frac{A_{in} - F + 2P}{S} \right\rfloor + 1 & \text{where $A \in \{W, H\}$} \\
\end{aligned}
$$

Therefore, the shape of the output $y$ is $(N, M, H_{out}, W_{out})$. The shape of the bias depends only on the number of feature maps $M$. Note that:

- The batch size of the output is the same as the batch size of the input. This is because each image in the batch is processed independently.
- The number of channels of the output is the same as the batch size of the kernel. This is because each output channel is produced by convolving the input with a different kernel.


### Implementation

Just as with the Linear Model, instead of viewing $y$ as $x \circledast w + b$, we can view it as `x -> Convolution -> Bias -> y`. That is, as a sequence of layers, each one transforming the input `x` until we obtain the output `y`.

In code terms, the `forward` method of a `Convolution` model is the composition of the `forward` functions of the `Convolution` and `Bias` layers:

```python
y_conv = conv2d(x)  # Implement the function inside the model itself
y = bias.forward(y_conv)
```

> TIP: true mathematical convolution (written $*$) flips the kernel; it equals the cross-correlation $\circledast$ between $x$ and $\text{rot}_{180^\circ } \left \{ w \right \}$, that is:
> $$
y = \left( \begin{bmatrix} x_{11} & x_{12} & x_{13} \\ x_{21} & x_{22} & x_{23} \\ x_{31} & x_{32} & x_{33} \end{bmatrix}  * \begin{bmatrix} w_{11} & w_{12} \\ w_{21} & w_{22} \end{bmatrix} \right)
\Leftrightarrow
\left( \begin{bmatrix} x_{11} & x_{12} & x_{13} \\ x_{21} & x_{22} & x_{23} \\ x_{31} & x_{32} & x_{33} \end{bmatrix} \circledast \begin{bmatrix} w_{22} & w_{21} \\ w_{12} & w_{11} \end{bmatrix} \right) = y
$$

Implement the `forward` method of the `Convolution` model in the file `edunn/models/convolution.py`.

In [ ]:
np.random.seed(123)

x = np.random.rand(2,3,7,7)
w = np.random.rand(4,3,5,5)
                       
stride,padding=2,1
kernel_initializer = nn.initializers.Constant(w)

layer=nn.Conv2d(x.shape[1],w.shape[0],kernel_size=w.shape[2:],stride=stride,padding=padding,kernel_initializer=kernel_initializer)

In [ ]:
y = layer.forward(x)
y, y.shape

## Backward Method

Knowing the gradient of the error function with respect to the output pixels, we must compute the total gradient of the error with respect to the input pixels and to the filter weights. Using the chain rule and partial derivatives we arrive at the expressions:

$$
\frac{\partial E}{\partial x_{mn}}=\sum_{ij}\frac{\partial E}{\partial y_{ij}}\frac{\partial y_{ij}}{\partial x_{mn}} 
\quad\quad\quad
\frac{\partial E}{\partial w_{mn}}=\sum_{ij}\frac{\partial E}{\partial y_{ij}}\frac{\partial y_{ij}}{\partial w_{mn}}
$$

Where each pixel $x_{mn}$ of the input and each pixel $w_{mn}$ of the filter contributes to one or more output pixels $y_{ij}$; for a given $x$ or $w$, the only output pixels that appear in the equations above are precisely those it contributed to during the forward computation.


### WITHOUT stride or padding

The computation is easiest to explain through an example. Suppose that $x \in \mathbb{R}^{(3\times 3)}$ and $w \in \mathbb{R}^{(2\times 2)}$, so that the $y_{ij}$ are defined as:

$$\begin{aligned}
y_{11} &= x_{11}w_{11} + x_{12}w_{12} + x_{21}w_{21} + x_{22}w_{22} \\
y_{12} &= x_{12}w_{11} + x_{13}w_{12} + x_{22}w_{21} + x_{23}w_{22} \\
y_{21} &= x_{21}w_{11} + x_{22}w_{12} + x_{31}w_{21} + x_{32}w_{22} \\
y_{22} &= x_{22}w_{11} + x_{23}w_{12} + x_{32}w_{21} + x_{33}w_{22}
\end{aligned}$$

#### `δE/δw`

The gradients of the error $E$ with respect to the filter $w$ (the vector of learned weights) can be computed by taking partial derivatives, exactly as explained in previous guides:

$$
\begin{aligned}
\frac{\partial E}{\partial w_{11}} &= \frac{\partial E}{\partial y_{11}} \frac{\partial y_{11}}{\partial w_{11}} + \frac{\partial E}{\partial y_{12}} \frac{\partial y_{12}}{\partial w_{11}} + \frac{\partial E}{\partial y_{21}} \frac{\partial y_{21}}{\partial w_{11}} + \frac{\partial E}{\partial y_{22}} \frac{\partial y_{22}}{\partial w_{11}}
\\
\frac{\partial E}{\partial w_{12}} &= \frac{\partial E}{\partial y_{11}} \frac{\partial y_{11}}{\partial w_{12}} + \frac{\partial E}{\partial y_{12}} \frac{\partial y_{12}}{\partial w_{12}} + \frac{\partial E}{\partial y_{21}} \frac{\partial y_{21}}{\partial w_{12}} + \frac{\partial E}{\partial y_{22}} \frac{\partial y_{22}}{\partial w_{12}}
\\
\frac{\partial E}{\partial w_{21}} &= \frac{\partial E}{\partial y_{11}} \frac{\partial y_{11}}{\partial w_{21}} + \frac{\partial E}{\partial y_{12}} \frac{\partial y_{12}}{\partial w_{21}} + \frac{\partial E}{\partial y_{21}} \frac{\partial y_{21}}{\partial w_{21}} + \frac{\partial E}{\partial y_{22}} \frac{\partial y_{22}}{\partial w_{21}}
\\
\frac{\partial E}{\partial w_{22}} &= \frac{\partial E}{\partial y_{11}} \frac{\partial y_{11}}{\partial w_{22}} + \frac{\partial E}{\partial y_{12}} \frac{\partial y_{12}}{\partial w_{22}} + \frac{\partial E}{\partial y_{21}} \frac{\partial y_{21}}{\partial w_{22}} + \frac{\partial E}{\partial y_{22}} \frac{\partial y_{22}}{\partial w_{22}}
\end{aligned}
$$

Using the equations defined at the beginning of this cell, it is easy to see that:

$$
\begin{aligned}
\frac{\partial E}{\partial w_{11}} &= \frac{\partial E}{\partial y_{11}} x_{11} + \frac{\partial E}{\partial y_{12}} x_{12} + \frac{\partial E}{\partial y_{21}} x_{21} + \frac{\partial E}{\partial y_{22}} x_{22}
\\
\frac{\partial E}{\partial w_{12}} &= \frac{\partial E}{\partial y_{11}} x_{12} + \frac{\partial E}{\partial y_{12}} x_{13} + \frac{\partial E}{\partial y_{21}} x_{22} + \frac{\partial E}{\partial y_{22}} x_{23}
\\
\frac{\partial E}{\partial w_{21}} &= \frac{\partial E}{\partial y_{11}} x_{21} + \frac{\partial E}{\partial y_{12}} x_{22} + \frac{\partial E}{\partial y_{21}} x_{31} + \frac{\partial E}{\partial y_{22}} x_{32}
\\
\frac{\partial E}{\partial w_{22}} &= \frac{\partial E}{\partial y_{11}} x_{22} + \frac{\partial E}{\partial y_{12}} x_{23} + \frac{\partial E}{\partial y_{21}} x_{32} + \frac{\partial E}{\partial y_{22}} x_{33}
\end{aligned}
$$

This follows exactly the same pattern as a convolution between $x$ and $\frac{\delta E}{\delta y}$:

$$
\frac{\delta E}{\delta w} = \begin{bmatrix} x_{11} & x_{12} & x_{13} \\ x_{21} & x_{22} & x_{23} \\ x_{31} & x_{32} & x_{33} \end{bmatrix}  \circledast \begin{bmatrix} \frac{\partial E}{\partial y_{11}} & \frac{\partial E}{\partial y_{12}} \\ \frac{\partial E}{\partial y_{21}} & \frac{\partial E}{\partial y_{22}} \end{bmatrix} = x \circledast \frac{\delta E}{\delta y}
$$

#### `δE/δx`

The reasoning for computing the gradients of $E$ with respect to the input image $x$ is the same as before:

$$
\begin{aligned}
\frac{\partial E}{\partial x_{11}}&=\frac{\partial E}{\partial y_{11}}w_{11}+\frac{\partial E}{\partial y_{12}}\;\:\, 0 \;+\frac{\partial E}{\partial y_{21}}\;\:\, 0 \;+\frac{\partial E}{\partial y_{22}}\;\:\, 0 \; 
\\
\frac{\partial E}{\partial x_{12}}&=\frac{\partial E}{\partial y_{11}}w_{12}+\frac{\partial E}{\partial y_{12}}w_{11}+\frac{\partial E}{\partial y_{21}}\;\:\, 0 \;+\frac{\partial E}{\partial y_{22}}\;\:\, 0 \; 
\\
\frac{\partial E}{\partial x_{13}}&=\frac{\partial E}{\partial y_{11}}\;\:\, 0 \;+\frac{\partial E}{\partial y_{12}}w_{12}+\frac{\partial E}{\partial y_{21}}\;\:\, 0 \;+\frac{\partial E}{\partial y_{22}}\;\:\, 0 \; 
\\
\frac{\partial E}{\partial x_{21}}&=\frac{\partial E}{\partial y_{11}}w_{21}+\frac{\partial E}{\partial y_{12}}\;\:\, 0 \;+\frac{\partial E}{\partial y_{21}}w_{11}+\frac{\partial E}{\partial y_{22}}\;\:\, 0 \; 
\\
\frac{\partial E}{\partial x_{22}}&=\frac{\partial E}{\partial y_{11}}w_{22}+\frac{\partial E}{\partial y_{12}}w_{21}+\frac{\partial E}{\partial y_{21}}w_{12}+\frac{\partial E}{\partial y_{22}}w_{11} 
\\
\frac{\partial E}{\partial x_{23}}&=\frac{\partial E}{\partial y_{11}}\;\:\, 0 \;+\frac{\partial E}{\partial y_{12}}w_{22}+\frac{\partial E}{\partial y_{21}}\;\:\, 0 \;+\frac{\partial E}{\partial y_{22}}w_{12} 
\\
\frac{\partial E}{\partial x_{31}}&=\frac{\partial E}{\partial y_{11}}\;\:\, 0 \;+\frac{\partial E}{\partial y_{12}}\;\:\, 0 \;+\frac{\partial E}{\partial y_{21}}w_{21}+\frac{\partial E}{\partial y_{22}}\;\:\, 0 \; 
\\
\frac{\partial E}{\partial x_{32}}&=\frac{\partial E}{\partial y_{11}}\;\:\, 0 \;+\frac{\partial E}{\partial y_{12}}\;\:\, 0 \;+\frac{\partial E}{\partial y_{21}}w_{22}+\frac{\partial E}{\partial y_{22}}w_{21} 
\\
\frac{\partial E}{\partial x_{33}}&=\frac{\partial E}{\partial y_{11}}\;\:\, 0 \;+\frac{\partial E}{\partial y_{12}}\;\:\, 0 \;+\frac{\partial E}{\partial y_{21}}\;\:\, 0 \;+\frac{\partial E}{\partial y_{22}}w_{22} 
\end{aligned}
$$

This computation can be automated with the **full-convolution** operator, in which the input is padded so that the output has the same dimensions as the input. Moreover, in order to express it that way, the filter matrix $w$ must be rotated by 180°. 

$$
\frac{\delta E}{\delta x} = \begin{bmatrix} \frac{\partial E}{\partial y_{11}} & \frac{\partial E}{\partial y_{12}} \\ \frac{\partial E}{\partial y_{21}} & \frac{\partial E}{\partial y_{22}} \end{bmatrix} \circledast \begin{bmatrix} w_{22} & w_{21} \\ w_{12} & w_{11} \end{bmatrix} = \left. \left[ \frac{\delta E}{\delta y} \circledast \text{rot}_{180^\circ } \left \{ w \right \} \right] \right|_{\text{full-padding=F-1}}
$$

![conv2d_backward.gif](img/conv2d_backward.gif)

The formula for the right amount of padding is easy to derive, and proving it is left as an exercise for the reader (you can find help in the [cs231n](https://cs231n.github.io/convolutional-networks/) blog).


### WITH stride and padding

The computation is easiest to explain through an example. Suppose that $x \in \mathbb{R}^{(5\times 5)}$ and $w \in \mathbb{R}^{(3\times 3)}$, with $S=2$, so that the $y_{ij}$ are defined as:

$$\begin{aligned}
y_{11} &=x_{11}w_{11}+x_{12}w_{12}+x_{13}w_{13}+x_{21}w_{21}+x_{22}w_{22}+x_{23}w_{23}+x_{31}w_{31}+x_{32}w_{32}+x_{33}w_{33} \\
y_{12} &=x_{13}w_{11}+x_{14}w_{12}+x_{15}w_{13}+x_{23}w_{21}+x_{24}w_{22}+x_{25}w_{23}+x_{33}w_{31}+x_{34}w_{32}+x_{35}w_{33} \\
y_{21} &=x_{31}w_{11}+x_{32}w_{12}+x_{33}w_{13}+x_{41}w_{21}+x_{42}w_{22}+x_{43}w_{23}+x_{51}w_{31}+x_{52}w_{32}+x_{53}w_{33} \\
y_{22} &=x_{33}w_{11}+x_{34}w_{12}+x_{35}w_{13}+x_{43}w_{21}+x_{44}w_{22}+x_{45}w_{23}+x_{53}w_{31}+x_{54}w_{32}+x_{55}w_{33} 
\end{aligned}$$

#### `δE/δx`

Because of the size of the input, the number of partial derivatives of $E$ with respect to the $x_{mn}$ is substantially larger than in the previous examples, so the derivation is hidden below and it is up to you whether to venture into that _inhospitable_ calculation.

<details><summary>Derivation</summary>

$$
\begin{aligned}
\frac{\partial E}{\partial x_{11}}&=
    \frac{\partial E}{\partial y_{11}}w_{11}+
    \frac{\partial E}{\partial y_{12}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{21}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{22}}\;\:\, 0 \; \\
\frac{\partial E}{\partial x_{12}}&=
    \frac{\partial E}{\partial y_{11}}w_{12}+
    \frac{\partial E}{\partial y_{12}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{21}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{22}}\;\:\, 0 \; \\
\frac{\partial E}{\partial x_{13}}&=
    \frac{\partial E}{\partial y_{11}}w_{13}+
    \frac{\partial E}{\partial y_{12}}w_{11}+
    \frac{\partial E}{\partial y_{21}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{22}}\;\:\, 0 \; \\
\frac{\partial E}{\partial x_{14}}&=
    \frac{\partial E}{\partial y_{11}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{12}}w_{12}+
    \frac{\partial E}{\partial y_{21}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{22}}\;\:\, 0 \; \\
\frac{\partial E}{\partial x_{15}}&=
    \frac{\partial E}{\partial y_{11}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{12}}w_{13}+
    \frac{\partial E}{\partial y_{21}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{22}}\;\:\, 0 \; \\
\frac{\partial E}{\partial x_{21}}&=
    \frac{\partial E}{\partial y_{11}}w_{21}+
    \frac{\partial E}{\partial y_{12}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{21}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{22}}\;\:\, 0 \; \\
\frac{\partial E}{\partial x_{22}}&=
    \frac{\partial E}{\partial y_{11}}w_{22}+
    \frac{\partial E}{\partial y_{12}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{21}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{22}}\;\:\, 0 \; \\
\frac{\partial E}{\partial x_{23}}&=
    \frac{\partial E}{\partial y_{11}}w_{23}+
    \frac{\partial E}{\partial y_{12}}w_{21}+
    \frac{\partial E}{\partial y_{21}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{22}}\;\:\, 0 \; \\
\frac{\partial E}{\partial x_{24}}&=
    \frac{\partial E}{\partial y_{11}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{12}}w_{22}+
    \frac{\partial E}{\partial y_{21}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{22}}\;\:\, 0 \; \\
\frac{\partial E}{\partial x_{25}}&=
    \frac{\partial E}{\partial y_{11}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{12}}w_{23}+
    \frac{\partial E}{\partial y_{21}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{22}}\;\:\, 0 \; \\
\frac{\partial E}{\partial x_{31}}&=
    \frac{\partial E}{\partial y_{11}}w_{31}+
    \frac{\partial E}{\partial y_{12}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{21}}w_{11}+
    \frac{\partial E}{\partial y_{22}}\;\:\, 0 \; \\
\frac{\partial E}{\partial x_{32}}&=
    \frac{\partial E}{\partial y_{11}}w_{32}+
    \frac{\partial E}{\partial y_{12}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{21}}w_{12}+
    \frac{\partial E}{\partial y_{22}}\;\:\, 0 \; \\
\frac{\partial E}{\partial x_{33}}&=
    \frac{\partial E}{\partial y_{11}}w_{33}+
    \frac{\partial E}{\partial y_{12}}w_{31}+
    \frac{\partial E}{\partial y_{21}}w_{13}+
    \frac{\partial E}{\partial y_{22}}w_{11} \\
\frac{\partial E}{\partial x_{34}}&=
    \frac{\partial E}{\partial y_{11}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{12}}w_{32}+
    \frac{\partial E}{\partial y_{21}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{22}}w_{12} \\
\frac{\partial E}{\partial x_{35}}&=
    \frac{\partial E}{\partial y_{11}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{12}}w_{33}+
    \frac{\partial E}{\partial y_{21}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{22}}w_{13} \\
\frac{\partial E}{\partial x_{41}}&=
    \frac{\partial E}{\partial y_{11}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{12}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{21}}w_{21}+
    \frac{\partial E}{\partial y_{22}}\;\:\, 0 \; \\
\frac{\partial E}{\partial x_{42}}&=
    \frac{\partial E}{\partial y_{11}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{12}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{21}}w_{22}+
    \frac{\partial E}{\partial y_{22}}\;\:\, 0 \; \\
\frac{\partial E}{\partial x_{43}}&=
    \frac{\partial E}{\partial y_{11}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{12}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{21}}w_{23}+
    \frac{\partial E}{\partial y_{22}}w_{21} \\
\frac{\partial E}{\partial x_{44}}&=
    \frac{\partial E}{\partial y_{11}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{12}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{21}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{22}}w_{22} \\
\frac{\partial E}{\partial x_{45}}&=
    \frac{\partial E}{\partial y_{11}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{12}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{21}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{22}}w_{23} \\
\frac{\partial E}{\partial x_{51}}&=
    \frac{\partial E}{\partial y_{11}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{12}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{21}}w_{31}+
    \frac{\partial E}{\partial y_{22}}\;\:\, 0 \; \\
\frac{\partial E}{\partial x_{52}}&=
    \frac{\partial E}{\partial y_{11}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{12}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{21}}w_{32}+
    \frac{\partial E}{\partial y_{22}}\;\:\, 0 \; \\
\frac{\partial E}{\partial x_{53}}&=
    \frac{\partial E}{\partial y_{11}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{12}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{21}}w_{33}+
    \frac{\partial E}{\partial y_{22}}w_{31} \\
\frac{\partial E}{\partial x_{54}}&=
    \frac{\partial E}{\partial y_{11}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{12}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{21}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{22}}w_{32} \\
\frac{\partial E}{\partial x_{55}}&=
    \frac{\partial E}{\partial y_{11}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{12}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{21}}\;\:\, 0 \;+
    \frac{\partial E}{\partial y_{22}}w_{33}
\end{aligned}
$$

</details>

These results follow a common pattern, which is easy to see if **$\frac{\partial E}{\partial y}$ is dilated by an amount equal to $S-1$**, then zero-padded to obtain a full-convolution, and the filter $w$ is rotated by 180°.

![conv2d_dilate.gif](img/conv2d_dilate.gif)

#### `δE/δw`

This case is handled very much like the previous one, and the pattern is again based on a convolution in which **$\frac{\partial E}{\partial y}$ must be dilated by an amount equal to $S-1$**, with no need for any padding at all.

$$
\begin{aligned}
\frac{\partial E}{\partial w_{11}}&=\frac{\partial E}{\partial y_{11}}x_{11}+\frac{\partial E}{\partial y_{12}}x_{13}+\frac{\partial E}{\partial y_{21}}x_{31}+\frac{\partial E}{\partial y_{22}}x_{33} \\
\frac{\partial E}{\partial w_{12}}&=\frac{\partial E}{\partial y_{11}}x_{12}+\frac{\partial E}{\partial y_{12}}x_{14}+\frac{\partial E}{\partial y_{21}}x_{32}+\frac{\partial E}{\partial y_{22}}x_{34} \\
\frac{\partial E}{\partial w_{13}}&=\frac{\partial E}{\partial y_{11}}x_{13}+\frac{\partial E}{\partial y_{12}}x_{15}+\frac{\partial E}{\partial y_{21}}x_{33}+\frac{\partial E}{\partial y_{22}}x_{35} \\
\frac{\partial E}{\partial w_{21}}&=\frac{\partial E}{\partial y_{11}}x_{21}+\frac{\partial E}{\partial y_{12}}x_{23}+\frac{\partial E}{\partial y_{21}}x_{41}+\frac{\partial E}{\partial y_{22}}x_{43} \\
\frac{\partial E}{\partial w_{22}}&=\frac{\partial E}{\partial y_{11}}x_{22}+\frac{\partial E}{\partial y_{12}}x_{24}+\frac{\partial E}{\partial y_{21}}x_{42}+\frac{\partial E}{\partial y_{22}}x_{44} \\
\frac{\partial E}{\partial w_{23}}&=\frac{\partial E}{\partial y_{11}}x_{23}+\frac{\partial E}{\partial y_{12}}x_{25}+\frac{\partial E}{\partial y_{21}}x_{43}+\frac{\partial E}{\partial y_{22}}x_{45} \\
\frac{\partial E}{\partial w_{31}}&=\frac{\partial E}{\partial y_{11}}x_{31}+\frac{\partial E}{\partial y_{12}}x_{33}+\frac{\partial E}{\partial y_{21}}x_{51}+\frac{\partial E}{\partial y_{22}}x_{53} \\
\frac{\partial E}{\partial w_{32}}&=\frac{\partial E}{\partial y_{11}}x_{32}+\frac{\partial E}{\partial y_{12}}x_{34}+\frac{\partial E}{\partial y_{21}}x_{52}+\frac{\partial E}{\partial y_{22}}x_{54} \\
\frac{\partial E}{\partial w_{33}}&=\frac{\partial E}{\partial y_{11}}x_{33}+\frac{\partial E}{\partial y_{12}}x_{35}+\frac{\partial E}{\partial y_{21}}x_{53}+\frac{\partial E}{\partial y_{22}}x_{55}
\end{aligned}
$$

> NOTE: keep in mind that in the backward computation the stride of the full-convolution is always set to 1, regardless of the stride used in the forward pass. This is because backpropagation computes the gradient of the error function with respect to every element of the input and of the kernel, so every position of the specific filter over each corresponding input pixel has to be taken into account.


In [ ]:
# Define the gradient of the output
g = np.random.rand(*y.shape)

# Backpropagate the gradient through the convolution
layer_grad = layer.backward(g)
layer_grad

## Checks against PyTorch

In [ ]:
import torch
import torch.nn as tnn

w = torch.from_numpy(w).to(torch.double)
x = torch.from_numpy(x).to(torch.double)

conv = tnn.Conv2d(in_channels=x.shape[1], out_channels=w.shape[0], kernel_size=w.shape[-1], stride=stride, padding=padding, bias=False)
conv.weight.data = w

x.requires_grad = True
conv.weight.requires_grad = True

y_torch = conv(x)
y_torch

In [ ]:
utils.check_same(y_torch.detach().numpy(),y)

In [ ]:
# Define the gradient of the output
g = torch.from_numpy(g).to(torch.double)

# Backpropagate the gradient through the convolution
y_torch.backward(g)

# Print the gradient of the input image and of the kernel
print("Gradient of the input (δE/δx):")
print(x.grad, x.grad.shape)
print("\nGradient of the kernel (δE/δw):")
print(conv.weight.grad, conv.weight.grad.shape)

In [ ]:
utils.check_same(x.grad.numpy(),layer_grad[0])

In [ ]:
utils.check_same(conv.weight.grad.numpy(),layer_grad[1]['w'])

In [ ]:
samples = 100
batch_size=2
din=3 # input dimension
dout=5 # output dimension
input_shape=(batch_size,din,5,5)

# Check the derivatives of a Convolution model
# with random values for `w`, `b`, and the input `x`
layer=nn.Conv2d(input_shape[1],dout,kernel_size=(3,3))


utils.check_gradient.common_layer(layer,input_shape,samples=samples,tolerance=1e-5)    

## Checks for 1D

### PyTorch

In [ ]:
import torch
import torch.nn as tnn

In [ ]:
stride,padding=1,0

In [ ]:
x1d = torch.randn(2, 3, 7, requires_grad=True)              # (batch_size, channels,    length)

x2d = torch.from_numpy(x1d.unsqueeze(2).detach().numpy())   # (batch_size, channels, 1, length)
x2d.requires_grad=True

In [ ]:
conv1d = tnn.Conv1d(x1d.shape[1], 4,     3 , stride=stride, padding=padding, bias=False)
conv2d = tnn.Conv2d(x1d.shape[1], 4, (1, 3), stride=stride, padding=padding, bias=False)
conv2d.weight.data = conv1d.weight.data.unsqueeze(2)

In [ ]:
output1d = conv1d(x1d)
output2d = conv2d(x2d)

In [ ]:
# (batch_size, num_filters, 1, new_length) -> (batch_size, num_filters, new_length)
output2d = output2d.squeeze(2)

In [ ]:
torch.allclose(output1d, output2d, atol=1e-6)

In [ ]:
grad_output = torch.randn_like(output1d)
grad_output_np = grad_output.unsqueeze(2).clone().numpy()

output1d.backward(grad_output.clone(), retain_graph=True)
output2d.backward(grad_output, retain_graph=True)

In [ ]:
torch.allclose(x1d.grad, x2d.grad.squeeze(2), atol=1e-6)

In [ ]:
torch.allclose(conv1d.weight.grad, conv2d.weight.grad.squeeze(2), atol=1e-6)

### edunn

In [ ]:
x = x2d.detach().numpy()
w = conv2d.weight.data.detach().numpy()
                       
kernel_initializer = nn.initializers.Constant(w)

layer=nn.Conv2d(x.shape[1],w.shape[0],kernel_size=w.shape[2:],stride=stride,padding=padding,kernel_initializer=kernel_initializer)

In [ ]:
y = layer.forward(x)
y, y.shape

In [ ]:
utils.check_same(output2d.detach().numpy(),y.squeeze(2), tol=1e-5)

In [ ]:
g = grad_output_np

layer_grad = layer.backward(g)
layer_grad

In [ ]:
utils.check_same(x2d.grad.squeeze(2).detach().numpy(),layer_grad[0].squeeze(2), tol=1e-5)

In [ ]:
utils.check_same(conv2d.weight.grad.squeeze(2).detach().numpy(),layer_grad[1]['w'].squeeze(2), tol=1e-5)